# 02 - Chest X-ray: Model Training


> **Academic prototype.** This notebook is part of a university final project.
> The models here are **not** medical devices, are **not** validated on clinical
> data, and must **never** be used to diagnose, screen or triage real patients.
> See `docs/ETHICS.md`.


**Goal:** train and compare two models on the split prepared in notebook 01.

1. **Baseline** - a small CNN trained from scratch (`configs/xray_baseline.yaml`)
2. **Transfer learning** - ImageNet-pretrained DenseNet-121 (`configs/xray_transfer.yaml`)

The baseline exists so the transfer-learning score has something honest to be
compared against. Both use the **same** split, labels and class weighting.

Model selection and early stopping use **validation** metrics only. The test set
is not touched here at all - that happens in notebook 03.

**Prerequisite:** `data/processed/xray_manifest.csv` from notebook 01.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

print("Project root:", PROJECT_ROOT)

In [ ]:
import pandas as pd
import torch

from src.common import get_device, load_config, seed_everything
from src.common.errors import DataNotFoundError
from src.common.io_utils import save_figure
from src.common.viz import plot_training_curves, set_plot_style
from src.classification import (
    build_dataloaders, build_loss, build_model, compute_pos_weights, peek_batch, train_model,
)

set_plot_style()

MANIFEST_PATH = PROJECT_ROOT / "data/processed/xray_manifest.csv"
if not MANIFEST_PATH.exists():
    raise DataNotFoundError(
        f"{MANIFEST_PATH} not found.\n"
        "  Fix: run 01_xray_eda_and_preparation.ipynb first - it creates this file."
    )

manifest = pd.read_csv(MANIFEST_PATH)
device = get_device("auto")
print(f"{len(manifest)} images | device={device}")
print(manifest["split"].value_counts().to_string())

## Quick-run switch

Set `SMOKE_TEST = True` to train for 1 epoch on a small subset. Use it to verify
the whole pipeline runs end-to-end **before** committing to a long run. Results
from a smoke test are not reportable - re-run with `False` for the real thing.

In [ ]:
SMOKE_TEST = False

OVERRIDES = {"train": {"epochs": 1, "batch_size": 8}} if SMOKE_TEST else {}

if SMOKE_TEST:
    # Shuffle then take the head of each split: same effect as a per-group
    # sample, without the pandas groupby.apply grouping-column warning.
    work_df = manifest.sample(frac=1.0, random_state=42).groupby("split").head(40)
    print(f"[smoke test] using {len(work_df)} images - results are NOT reportable")
else:
    work_df = manifest

---
# Part A - Baseline CNN (from scratch)

In [ ]:
cfg_base = load_config("xray_baseline.yaml", overrides=OVERRIDES)
seed_everything(cfg_base.get("seed", 42), deterministic=cfg_base.get("deterministic", True))

LABELS = list(cfg_base.data.labels)
loaders_base = build_dataloaders(work_df, cfg_base)

In [ ]:
# Sanity check one batch before training: shapes, value range, positive rate.
_ = peek_batch(loaders_base["train"])

### Class imbalance handling

`pos_weight = n_negative / n_positive`, computed on the **training split only** (using val/test statistics would leak). It multiplies the loss contribution of positive examples so the rare class is not ignored.

In [ ]:
pos_weight = compute_pos_weights(work_df, LABELS, split="train")
criterion_base = build_loss(cfg_base, pos_weight=pos_weight, device=device)

In [ ]:
model_base = build_model(cfg_base, num_labels=len(LABELS))

### Train the baseline

Everything (checkpoints, history CSV, config snapshot) is written to `outputs/classification/<run_name>/`.

In [ ]:
result_base = train_model(model_base, loaders_base, criterion_base, cfg_base, device)

In [ ]:
fig = plot_training_curves(result_base["history"], metrics=["loss", "roc_auc"],
                           title="Baseline CNN - training curves")
save_figure(fig, result_base["dirs"]["figures"] / "training_curves.png", close=False)
result_base["history"].round(4)

---
# Part B - Transfer learning (DenseNet-121)

Same data, same split, same class weighting - only the model changes. The first run downloads ImageNet weights (~30 MB) and needs an internet connection.

Note the much smaller learning rate (`1e-4` vs `1e-3`): the backbone already contains useful features and large updates would destroy them.

In [ ]:
cfg_tl = load_config("xray_transfer.yaml", overrides=OVERRIDES)
seed_everything(cfg_tl.get("seed", 42), deterministic=cfg_tl.get("deterministic", True))

assert list(cfg_tl.data.labels) == LABELS, (
    "labels differ between the two configs - the comparison would not be fair")

loaders_tl = build_dataloaders(work_df, cfg_tl)
criterion_tl = build_loss(cfg_tl, pos_weight=pos_weight, device=device)
model_tl = build_model(cfg_tl, num_labels=len(LABELS))

In [ ]:
result_tl = train_model(model_tl, loaders_tl, criterion_tl, cfg_tl, device)

In [ ]:
fig = plot_training_curves(result_tl["history"], metrics=["loss", "roc_auc"],
                           title=f"{cfg_tl.get('model.name')} - training curves")
save_figure(fig, result_tl["dirs"]["figures"] / "training_curves.png", close=False)
result_tl["history"].round(4)

## Validation comparison

Best **validation** score of each run. This is a model-selection table, not a result table - the reportable comparison is on the test set in notebook 03.

In [ ]:
comparison = pd.DataFrame([
    {"run": cfg_base.get("run_name"), "model": cfg_base.get("model.name"),
     "pretrained": cfg_base.get("model.pretrained"),
     "best_epoch": result_base["best_epoch"], "best_val_roc_auc": result_base["best_score"],
     "checkpoint": str(result_base["best_checkpoint"].name)},
    {"run": cfg_tl.get("run_name"), "model": cfg_tl.get("model.name"),
     "pretrained": cfg_tl.get("model.pretrained"),
     "best_epoch": result_tl["best_epoch"], "best_val_roc_auc": result_tl["best_score"],
     "checkpoint": str(result_tl["best_checkpoint"].name)},
])
comparison.round(4)

In [ ]:
print("Checkpoints for notebook 03:")
print("  baseline:", result_base["best_checkpoint"])
print("  transfer:", result_tl["best_checkpoint"])

---

## Notes for the report

- Compare the two **training curves**: a from-scratch CNN on a small dataset
  usually shows a widening train/validation gap (overfitting) much earlier than
  the pretrained model.
- If validation loss rises while training loss keeps falling, early stopping did
  its job - say so, and quote the epoch it stopped at.
- Record the actual epochs, batch size and learning rate you ran. Do not copy the
  config defaults if you changed them.

### Screenshots for the report
- Both training-curve figures
- The class-imbalance / `pos_weight` output
- The validation comparison table

**Next:** `03_xray_evaluation_and_gradcam.ipynb`